# 03 APP / mule simulation layer
Why simulate: there is no credible open APP-labelled dataset with device, session and beneficiary signals. The simulator (`src/presage/simulation/app_world.py`) is a discrete-event model. **Everything it outputs is synthetic.**

We run a tiny world here to see the mechanics; the full world (`sim_v1`) is built by `make data`.

In [1]:
import sys, pathlib, json, warnings; warnings.filterwarnings('ignore')
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 40)
plt.rcParams.update({'figure.dpi': 100, 'axes.spines.top': False, 'axes.spines.right': False})
def table(name):
    p = ROOT / 'reports' / 'tables' / name
    return pd.read_csv(p) if p.exists() else None

In [2]:
from presage.simulation.app_world import simulate, SimConfig
small = SimConfig(n_customers=800, n_payees=1500, days=60, n_mule_networks=20, scam_rate_per_day=2.0, recurring_windows=((10, 20),), sudden_drift_day=45)
df, acc = simulate(small); print(df.shape); df.head(3)

18:45:09 INFO sim: simulated 41038 txns (222 scam payments = 0.541%), 2515 accounts, 215 mule accounts


(41038, 25)


,txn_id,ts,src,dst,amount,txn_type,customer_id,device_id,ip_id,session_s,failed_ben_adds,channel,scam_type,campaign_id,net_id,report_delay_s,is_app_fraud,balance_before,bank_src,bank_dst,cross_bank,dst_age_days,src_age_days,day,src_is_customer
0,0,158.346561,477,1855,52.706939,legit,477,1431,850,130.961060,0,mobile,,-1,-1,NaN,0,5561.119181,0,2,1,626.713633,19.861080,0,1
1,1,228.977034,205,1060,98.689075,legit,205,616,677,130.110243,0,desktop,,-1,-1,NaN,0,16994.996934,3,0,1,40.286679,401.492388,0,1
2,2,687.188125,556,1161,19.487466,legit,556,1668,436,36.190766,0,desktop,,-1,-1,NaN,0,7749.786161,0,2,1,845.613128,69.397173,0,1


### Anatomy of one scam
A victim (genuine, authenticated) pays a tier-1 mule; the mule forwards funds to tier-2 after a random holding time, then tier-3 cashes out.

In [3]:
net = df[df.is_app_fraud==1].net_id.value_counts().index[0]
flow = df[df.net_id==net].sort_values('ts')[['ts','src','dst','amount','txn_type','scam_type']]
flow['hours_since_first'] = ((flow.ts - flow.ts.min())/3600).round(1); flow.head(12)

,ts,src,dst,amount,txn_type,scam_type,hours_since_first
7185,9.780232e+05,303,2434,3765.815726,scam,impersonation,0.0
7478,1.005431e+06,169,2434,3831.517367,scam,bank_impersonation,7.6
7616,1.015548e+06,256,2433,148.523909,scam,purchase,10.4
7639,1.018200e+06,476,2433,6217.418573,scam,bank_impersonation,11.2
7802,1.050625e+06,2434,2435,1263.413391,mule_forward,,20.2
7805,1.051225e+06,2434,2435,1263.413391,mule_forward,,20.3
7807,1.051825e+06,2434,2435,1263.413391,mule_forward,,20.5
7861,1.065444e+06,540,2434,617.382844,scam,family_impersonation,24.3
7884,1.067174e+06,2434,2439,1190.028202,mule_forward,,24.8
7889,1.067774e+06,2434,2439,1190.028202,mule_forward,,24.9


### What is original vs injected
| Element | Origin |
|---|---|
| All rows, accounts, devices, IPs, sessions | synthetic |
| Real datasets (PaySim, IBM AML, Elliptic) | untouched, used separately; never blended into the simulation |
| Scam signals (new beneficiary, session length, failed adds, structuring, mule fan-in/out, fast pass-through) | injected by typology parameters |
Parameters are plausible assumptions, not calibrated to a bank.

In [4]:
sig = df[df.src_is_customer==1].assign(scam=lambda d: d.is_app_fraud==1).groupby('scam').agg(median_amount=('amount','median'), mean_session=('session_s','mean'), mean_failed=('failed_ben_adds','mean'), dst_age_median=('dst_age_days','median'))
sig

,median_amount,mean_session,mean_failed,dst_age_median
scam,,,,
False,42.970745,90.318994,0.027699,307.380991
True,504.168706,210.928162,0.373874,195.888914


In [5]:
from presage.simulation.app_world import SCAM_TYPES; print(SCAM_TYPES)

['investment', 'purchase', 'romance', 'impersonation', 'bank_impersonation', 'invoice_redirection', 'ceo_fraud', 'family_impersonation']
